# Multi-GPU batched singular-value comparison

This notebook measures **strong scaling** for singular-values-only computation on real square matrices by splitting the batch dimension across multiple GPUs.

It uses the same wall-clock + explicit-synchronization timing convention as the single-GPU notebook and tests both `float32` and `float64`. Compilation/warm-up is excluded. Backends are included only when they expose enough GPU devices; once included, runtime failures are not hidden by blanket exception handling.

The notebook can compare CuPy, PyTorch, JAX and the custom PyCUDA implementation. NumPy is used only as the accuracy reference because it does not provide a multi-GPU path.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None


## Configuration

In [ ]:
matrix_size = 4
dtypes = [np.float32, np.float64]
total_batch_sizes = [8192, 32768, 131072]
repeats = 5
seed = 54321
max_gpus_to_test = None   # None = use every available GPU


In [ ]:
@dataclass
class TimingStats:
    median: float
    minimum: float
    maximum: float
    mean: float
    std: float


def summarize_times(samples):
    x = np.asarray(samples, dtype=np.float64)
    return TimingStats(float(np.median(x)), float(np.min(x)), float(np.max(x)),
                       float(np.mean(x)), float(np.std(x)))


def split_slices(B, k):
    counts = np.full(k, B // k, dtype=int)
    counts[:B % k] += 1
    slices = []
    start = 0
    for count in counts:
        slices.append(slice(start, start + int(count)))
        start += int(count)
    return slices


def make_batch(B, dtype):
    rng = np.random.default_rng(seed + B)
    return rng.standard_normal((B, matrix_size, matrix_size)).astype(dtype)


def validation_metrics(values, reference):
    values = np.sort(np.asarray(values, dtype=np.float64), axis=1)[:, ::-1]
    reference = np.sort(np.asarray(reference, dtype=np.float64), axis=1)[:, ::-1]
    nonfinite = int(values.size - np.count_nonzero(np.isfinite(values)))
    if nonfinite:
        return nonfinite, np.inf, np.inf
    denom = np.maximum(np.linalg.norm(reference, axis=1), np.finfo(float).tiny)
    rel = np.linalg.norm(values-reference, axis=1) / denom
    return 0, float(np.mean(rel)), float(np.max(rel))


## Detect GPU backends

In [ ]:
availability = {}

try:
    import cupy as cp
except ImportError as exc:
    availability['CuPy'] = (0, f'not installed: {exc}')
else:
    try:
        count = cp.cuda.runtime.getDeviceCount()
    except cp.cuda.runtime.CUDARuntimeError as exc:
        availability['CuPy'] = (0, f'CUDA unavailable: {exc}')
    else:
        availability['CuPy'] = (count, f'{count} CUDA device(s)')

try:
    import torch
except ImportError as exc:
    availability['PyTorch'] = (0, f'not installed: {exc}')
else:
    count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    availability['PyTorch'] = (count, f'{count} CUDA device(s)' if count else 'CUDA unavailable')

try:
    import jax
except ImportError as exc:
    availability['JAX'] = (0, f'not installed: {exc}')
else:
    jax.config.update('jax_enable_x64', True)
    jax_gpu_devices = [d for d in jax.devices() if d.platform == 'gpu']
    availability['JAX'] = (len(jax_gpu_devices), f'{len(jax_gpu_devices)} GPU device(s)')

try:
    import pycuda.driver as drv
except ImportError as exc:
    availability['Custom PyCUDA'] = (0, f'not installed: {exc}')
else:
    try:
        drv.init(); count = drv.Device.count()
    except drv.Error as exc:
        availability['Custom PyCUDA'] = (0, f'CUDA unavailable: {exc}')
    else:
        availability['Custom PyCUDA'] = (count, f'{count} CUDA device(s)')

for name, (count, reason) in availability.items():
    print(f'{name:14s}: {reason}')


## Multi-GPU backend implementations

In [ ]:
def benchmark_cupy_multi(batch, k):
    slices = split_slices(len(batch), k)
    devices = list(range(k))

    def upload_worker(dev, sl):
        with cp.cuda.Device(dev):
            return cp.asarray(batch[sl])

    def compute_worker(dev, x):
        with cp.cuda.Device(dev):
            y = cp.linalg.svd(x, full_matrices=False, compute_uv=False)
            cp.cuda.Device(dev).synchronize()
            return y

    def e2e_worker(dev, sl):
        with cp.cuda.Device(dev):
            x = cp.asarray(batch[sl])
            y = cp.linalg.svd(x, full_matrices=False, compute_uv=False)
            out = cp.asnumpy(y)
            cp.cuda.Device(dev).synchronize()
            return out

    with ThreadPoolExecutor(max_workers=k) as pool:
        xs = list(pool.map(upload_worker, devices, slices))
        _ = list(pool.map(compute_worker, devices, xs))  # warm-up

        device_samples = []
        ys = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            ys = list(pool.map(compute_worker, devices, xs))
            device_samples.append(time.perf_counter() - t0)
        values = np.vstack([cp.asnumpy(y) for y in ys])

        _ = list(pool.map(e2e_worker, devices, slices))  # warm-up
        e2e_samples = []
        outs = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            outs = list(pool.map(e2e_worker, devices, slices))
            e2e_samples.append(time.perf_counter() - t0)
    return np.vstack(outs), summarize_times(device_samples), summarize_times(e2e_samples)


In [ ]:
def benchmark_torch_multi(batch, k):
    slices = split_slices(len(batch), k)
    devices = list(range(k))
    tdtype = torch.float32 if batch.dtype == np.float32 else torch.float64

    def upload_worker(dev, sl):
        with torch.cuda.device(dev):
            x = torch.as_tensor(batch[sl], dtype=tdtype, device=f'cuda:{dev}')
            torch.cuda.synchronize(dev)
            return x

    def compute_worker(dev, x):
        with torch.cuda.device(dev):
            y = torch.linalg.svdvals(x)
            torch.cuda.synchronize(dev)
            return y

    def e2e_worker(dev, sl):
        with torch.cuda.device(dev):
            x = torch.as_tensor(batch[sl], dtype=tdtype, device=f'cuda:{dev}')
            y = torch.linalg.svdvals(x)
            out = y.detach().cpu().numpy()
            torch.cuda.synchronize(dev)
            return out

    with ThreadPoolExecutor(max_workers=k) as pool:
        xs = list(pool.map(upload_worker, devices, slices))
        _ = list(pool.map(compute_worker, devices, xs))

        device_samples = []
        ys = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            ys = list(pool.map(compute_worker, devices, xs))
            device_samples.append(time.perf_counter() - t0)
        values = np.vstack([y.detach().cpu().numpy() for y in ys])

        _ = list(pool.map(e2e_worker, devices, slices))
        e2e_samples = []
        outs = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            outs = list(pool.map(e2e_worker, devices, slices))
            e2e_samples.append(time.perf_counter() - t0)
    return np.vstack(outs), summarize_times(device_samples), summarize_times(e2e_samples)


In [ ]:
if availability['JAX'][0] > 0:
    import jax.numpy as jnp

    @jax.jit
    def _jax_svals_multi(x):
        return jnp.linalg.svd(x, full_matrices=False, compute_uv=False)

    def benchmark_jax_multi(batch, k):
        slices = split_slices(len(batch), k)
        devices = jax_gpu_devices[:k]
        xs = [jax.device_put(batch[sl], dev) for sl, dev in zip(slices, devices)]
        ys = [_jax_svals_multi(x) for x in xs]
        for y in ys: y.block_until_ready()  # warm-up every shard shape/device

        device_samples = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            ys = [_jax_svals_multi(x) for x in xs]
            for y in ys: y.block_until_ready()
            device_samples.append(time.perf_counter() - t0)
        values = np.vstack([np.asarray(y) for y in ys])

        # End-to-end warm-up.
        ys2 = [_jax_svals_multi(jax.device_put(batch[sl], dev)) for sl, dev in zip(slices, devices)]
        _ = [np.asarray(y) for y in ys2]
        e2e_samples = []
        outs = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            ys2 = [_jax_svals_multi(jax.device_put(batch[sl], dev)) for sl, dev in zip(slices, devices)]
            outs = [np.asarray(y) for y in ys2]
            e2e_samples.append(time.perf_counter() - t0)
        return np.vstack(outs), summarize_times(device_samples), summarize_times(e2e_samples)


In [ ]:
if availability['Custom PyCUDA'][0] > 0:
    from custom_pycuda_svals import MultiGPUCustomBatchedSingularValues

    def benchmark_custom_multi(batch, k):
        with MultiGPUCustomBatchedSingularValues(
            len(batch), matrix_size, batch.dtype, device_ids=range(k)
        ) as solver:
            device = solver.benchmark_device(batch, repeats=repeats)
            e2e = solver.benchmark_end_to_end(batch, repeats=repeats)
        return e2e.singular_values, device.timing, e2e.timing


## Run strong-scaling comparison

In [ ]:
backend_functions = {}
if availability['CuPy'][0] > 0: backend_functions['CuPy'] = benchmark_cupy_multi
if availability['PyTorch'][0] > 0: backend_functions['PyTorch'] = benchmark_torch_multi
if availability['JAX'][0] > 0: backend_functions['JAX'] = benchmark_jax_multi
if availability['Custom PyCUDA'][0] > 0: backend_functions['Custom PyCUDA'] = benchmark_custom_multi

rows = []
for dtype in dtypes:
    for B in total_batch_sizes:
        batch = make_batch(B, dtype)
        reference = np.linalg.svd(batch.astype(np.float64), compute_uv=False)
        for backend, fn in backend_functions.items():
            available_count = availability[backend][0]
            limit = available_count if max_gpus_to_test is None else min(available_count, max_gpus_to_test)
            for k in range(1, limit + 1):
                values, device_stats, e2e_stats = fn(batch, k)
                nonfinite, mean_err, max_err = validation_metrics(values, reference)
                rows.append({
                    'dtype': np.dtype(dtype).name,
                    'backend': backend,
                    'batch_size': B,
                    'gpu_count': k,
                    'device_median_s': device_stats.median,
                    'end_to_end_median_s': e2e_stats.median,
                    'matrices_per_second_device': B / device_stats.median,
                    'matrices_per_second_e2e': B / e2e_stats.median,
                    'nonfinite_count': nonfinite,
                    'mean_relative_error': mean_err,
                    'max_relative_error': max_err,
                })
                print(f'{np.dtype(dtype).name:7s} B={B:7d} {backend:14s} GPUs={k}: '
                      f'device={device_stats.median:.4e}s e2e={e2e_stats.median:.4e}s '
                      f'maxerr={max_err:.3e}')

if not rows:
    print('No GPU backend is available in this runtime. The notebook is otherwise ready for a multi-GPU CUDA system.')


In [ ]:
if pd is not None:
    results = pd.DataFrame(rows)
    if len(results):
        # Strong-scaling speedup relative to the same backend/dtype/batch on one GPU.
        results['device_speedup'] = np.nan
        results['e2e_speedup'] = np.nan
        for keys, idx in results.groupby(['dtype', 'backend', 'batch_size']).groups.items():
            group = results.loc[idx]
            base = group[group['gpu_count'] == 1]
            if len(base) == 1:
                base_d = float(base['device_median_s'].iloc[0])
                base_e = float(base['end_to_end_median_s'].iloc[0])
                results.loc[idx, 'device_speedup'] = base_d / group['device_median_s']
                results.loc[idx, 'e2e_speedup'] = base_e / group['end_to_end_median_s']
        display(results)
else:
    results = rows
    for row in rows: print(row)


In [ ]:
if pd is not None and len(results):
    for dtype_name in results['dtype'].unique():
        for B in results['batch_size'].unique():
            sub = results[(results['dtype'] == dtype_name) & (results['batch_size'] == B)]
            plt.figure(figsize=(8, 5))
            for backend, group in sub.groupby('backend'):
                plt.plot(group['gpu_count'], group['device_speedup'], marker='o', label=backend)
            max_k = int(sub['gpu_count'].max())
            plt.plot(range(1, max_k + 1), range(1, max_k + 1), '--', label='ideal')
            plt.xlabel('Number of GPUs')
            plt.ylabel('Device-resident speedup')
            plt.title(f'Strong scaling - {dtype_name}, batch={B}')
            plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()


## Notes on interpretation

- Strong scaling keeps the **total batch size fixed** while changing the number of GPUs.
- Thread-pool orchestration is used for CuPy and PyTorch so blocking host calls on one device do not serialize work intended for another device.
- JAX work is dispatched to explicit devices and then synchronized collectively.
- The custom PyCUDA path keeps one primary context and one stream per device and enqueues work on all devices before synchronization.
- Multi-GPU end-to-end timing includes host/device data movement. It can therefore scale less well than device-resident compute, especially for small batches.